# Kisan Mitra — live demo (Gradio)

Launches the web app with a **public share link** so you can demo it from any device.

1. *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
2. Open the `https://xxxx.gradio.live` link printed at the bottom.
3. Try the example questions (Hindi / Kannada / Tamil / English), the microphone, and "Read the answer aloud".
4. For the submission: take screenshots (one per language + the fertilizer tool + a refusal) and record a
   2–3 minute screen video. Save them into `docs/screenshots/` of the project.

The link stays alive while this notebook is running (max ~72 h, usually until Colab idles out).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Get the project: from Google Drive if you uploaded it there, otherwise clone it from GitHub
import os, sys, subprocess
REPO_URL = 'https://github.com/amanchandrah895/kisan-mitra.git'   # used only if the Drive folder is not found
DRIVE_DIR = '/content/drive/MyDrive/kisan-mitra'

try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
except Exception as e:
    print('Drive not mounted:', e)

if os.path.exists(f'{DRIVE_DIR}/configs/default.yaml'):
    PROJECT_DIR = DRIVE_DIR                      # results persist on Drive
else:
    PROJECT_DIR = '/content/kisan-mitra'
    if not os.path.exists(f'{PROJECT_DIR}/configs/default.yaml'):
        subprocess.run(['git', 'clone', '-q', REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
if f'{PROJECT_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{PROJECT_DIR}/src')
# results go to Drive whenever it is mounted (survives Colab disconnects)
if PROJECT_DIR == DRIVE_DIR:
    RESULTS_DIR = f'{DRIVE_DIR}/results'
elif os.path.exists('/content/drive/MyDrive'):
    RESULTS_DIR = '/content/drive/MyDrive/kisan-mitra-results/results'
else:
    RESULTS_DIR = RESULTS_DIR
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Project folder:', PROJECT_DIR, '| results ->', RESULTS_DIR)

In [ ]:
# 3) Install dependencies (~2-3 min). Colab already has torch / torchaudio.
!pip install -q -r requirements.txt
!pip install -q -r requirements-asr.txt || pip install -q onnxruntime-gpu onnx
!apt-get -qq install -y ffmpeg > /dev/null
print('\nDone. If pip printed "you must restart the runtime": Runtime > Restart session, then run cell 2 again and continue from cell 4.')

In [ ]:
# 4) Hugging Face token + model access check
#    - Add your token: left sidebar "key" icon (Secrets) > Add new secret > Name: HF_TOKEN, Value: hf_xxx > enable "Notebook access"
#    - Accept the terms (one click each, while logged in) on:
#        https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M
#        https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M
#        https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual
from kisan_mitra.config import load_config
from kisan_mitra.hf_utils import hf_token, check_model_access, env_report

cfg = load_config()
print('HF token found' if hf_token() else 'NO HF_TOKEN FOUND - add it in Colab Secrets (see above)')
status = check_model_access(list(cfg['models'].values()))
for m, s in status.items():
    print(('OK   ' if s == 'ok' else 'FAIL ') + m + ('' if s == 'ok' else '  ->  ' + s))
print(env_report())
bad = [m for m, s in status.items() if s != 'ok']
assert not bad, 'Fix model access for: ' + ', '.join(bad)

In [ ]:
# 5) Load the models (AgriParam + IndicTrans2 + retriever + grounding checker + IndicConformer ASR), ~5-8 min first time
from kisan_mitra.app import build_agent_for_app, build_app
agent = build_agent_for_app()
r = agent.answer('5 एकड़ धान के लिए कितना यूरिया चाहिए?')
print(r['route'], '|', r['answer'])

In [ ]:
# 6) Launch - click the public URL
demo = build_app(agent)
demo.launch(share=True, debug=True)